In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config
from lib_etl.utility import *

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

### Transform 

In [0]:
df_segment = spark.sql(f"""
    SELECT 
        cast(AGE_RANGE as string) as AGE_RANGE,
        cast(dist_range as string) as dist_range,
        cast(`25_Percentile_annual_sales` as double) as `25_Percentile_annual_sales`,
        cast(`50_Percentile_annual_sales` as double) as `50_Percentile_annual_sales`,
        cast(`75_Percentile_annual_sales` as double) as `75_Percentile_annual_sales`,
        cast(Avg_annual_sales as double) as Avg_annual_sales,
        cast(Count as long) as Count,
        cast(Median_annual_sales as double) as Median_annual_sales,
        cast(Strategic_segment as string) as Strategic_segment,
        cast(dist_lower as double) as dist_lower,
        cast(dist_upper as double) as dist_upper,
        cast(age_lower as double) as age_lower,
        cast(age_upper as double) as age_upper
    FROM 
        {bronze_bcg_maps_strategic_segments}
""").dropDuplicates()

df_segment.createOrReplaceTempView("source")

### Merge

In [0]:
df_segment.write.mode("overwrite").saveAsTable(silver_bcg_maps_strategic_segments)